In [1]:
# Cell 1
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)

df = pd.read_csv('../data/application_train.csv')

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print(f"\nTarget distribution:")
print(df['TARGET'].value_counts())

Rows    : 307,511
Columns : 122

Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64


In [2]:
# Cell 2 - Drop columns with more than 50% missing values

# Calculate missing percentage for each column
missing_pct = df.isnull().mean()

# Protect EXT_SOURCE columns — we handle them specially in Cell 3
protect_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

# Drop columns with >50% missing EXCEPT protected ones
cols_to_drop = missing_pct[
    (missing_pct > 0.50) &
    (~missing_pct.index.isin(protect_cols))
].index.tolist()

print(f"Columns before dropping  : {df.shape[1]}")
print(f"Columns to drop          : {len(cols_to_drop)}")
print(f"EXT_SOURCE cols protected: ✅")

df = df.drop(columns=cols_to_drop)

print(f"Columns after dropping   : {df.shape[1]}")

# Confirm EXT_SOURCE columns are present
for col in protect_cols:
    print(f"  {col} present: {col in df.columns}")

Columns before dropping  : 122
Columns to drop          : 40
EXT_SOURCE cols protected: ✅
Columns after dropping   : 82
  EXT_SOURCE_1 present: True
  EXT_SOURCE_2 present: True
  EXT_SOURCE_3 present: True


In [3]:
# Cell 3 - Fix EXT_SOURCE columns

ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

# Check missing in each
for col in ext_cols:
    n = df[col].isnull().sum()
    print(f"{col} missing: {n:,} ({n/len(df):.1%})")

# Flag rows where ALL THREE are missing
df['is_missing_ext'] = df[ext_cols].isnull().all(axis=1).astype(int)

# Compute mean across available scores — ignores NaN automatically
df['mean_ext_score'] = df[ext_cols].mean(axis=1, skipna=True)

# Where all three missing, mean will be NaN — fill with 0
df['mean_ext_score'] = df['mean_ext_score'].fillna(0)

# Drop the original three columns
df = df.drop(columns=ext_cols)

print(f"\nis_missing_ext = 1 : {df['is_missing_ext'].sum():,} rows")
print(f"mean_ext_score NaN : {df['mean_ext_score'].isnull().sum()}")
print(f"mean_ext_score range : {df['mean_ext_score'].min():.4f} → {df['mean_ext_score'].max():.4f}")
print(f"\nShape : {df.shape}")

EXT_SOURCE_1 missing: 173,378 (56.4%)
EXT_SOURCE_2 missing: 660 (0.2%)
EXT_SOURCE_3 missing: 60,965 (19.8%)

is_missing_ext = 1 : 172 rows
mean_ext_score NaN : 0
mean_ext_score range : 0.0000 → 0.8789

Shape : (307511, 81)


In [4]:
# Cell 4 - Fix DAYS_EMPLOYED anomaly

# Step 1 — identify who has the anomaly
anomaly_mask = df['DAYS_EMPLOYED'] == 365243

print(f"Total anomaly rows : {anomaly_mask.sum():,}")
print(f"\nBreakdown by NAME_INCOME_TYPE:")
print(df[anomaly_mask]['NAME_INCOME_TYPE'].value_counts())

# Step 2 — replace all 365243 with NaN first
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

# Step 3 — for pensioners and unemployed → fill with 0
# They are not employed — 0 is the honest value
not_employed = ['Pensioner', 'Unemployed', 'Student', 'Maternity leave']
not_employed_mask = df['NAME_INCOME_TYPE'].isin(not_employed)

df.loc[not_employed_mask, 'DAYS_EMPLOYED'] = df.loc[
    not_employed_mask, 'DAYS_EMPLOYED'
].fillna(0)

print(f"\nAfter filling not-employed with 0:")
print(f"  NaN remaining : {df['DAYS_EMPLOYED'].isnull().sum():,}")

# Step 4 — remaining NaN are working people with data errors → fill with median
median_val = df['DAYS_EMPLOYED'].median()
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].fillna(median_val)

print(f"  After median fill : {df['DAYS_EMPLOYED'].isnull().sum()}")
print(f"  Median used       : {median_val:.0f} days ({median_val/-365:.1f} years)")
print(f"  Max value now     : {df['DAYS_EMPLOYED'].max():,}")
print(f"  Min value now     : {df['DAYS_EMPLOYED'].min():,}")

Total anomaly rows : 55,374

Breakdown by NAME_INCOME_TYPE:
NAME_INCOME_TYPE
Pensioner     55352
Unemployed       22
Name: count, dtype: int64

After filling not-employed with 0:
  NaN remaining : 0
  After median fill : 0
  Median used       : -1213 days (3.3 years)
  Max value now     : 0.0
  Min value now     : -17,912.0


In [8]:
# Cell 5

# Age in years (DAYS_BIRTH is negative)
df['AGE_YEARS'] = df['DAYS_BIRTH'] / -365

# Years employed (DAYS_EMPLOYED already cleaned)
df['EMPLOYMENT_YEARS'] = df['DAYS_EMPLOYED'] / -365

# What fraction of their life have they been employed
df['EMPLOYMENT_AGE_RATIO'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']

# Can they afford the loan
df['INCOME_CREDIT_RATIO'] = df['AMT_INCOME_TOTAL'] / df['AMT_CREDIT']

# What % of income goes to repayment
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']

# Income spread across family
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

# Implied loan term
df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']

# Drop raw day columns — replaced by cleaner features
df = df.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED'])

print("New features created:")
new_cols = ['AGE_YEARS', 'EMPLOYMENT_YEARS', 'EMPLOYMENT_AGE_RATIO',
            'INCOME_CREDIT_RATIO', 'ANNUITY_INCOME_RATIO',
            'INCOME_PER_PERSON', 'CREDIT_TERM']

for col in new_cols:
    print(f"  {col:<25} NaN: {df[col].isnull().sum():,}")

print(f"\nShape : {df.shape}")

New features created:
  AGE_YEARS                 NaN: 0
  EMPLOYMENT_YEARS          NaN: 0
  EMPLOYMENT_AGE_RATIO      NaN: 0
  INCOME_CREDIT_RATIO       NaN: 0
  ANNUITY_INCOME_RATIO      NaN: 12
  INCOME_PER_PERSON         NaN: 2
  CREDIT_TERM               NaN: 12

Shape : (307511, 86)


In [9]:
# Cell 6 - Impute all remaining missing values

# Separate numeric and categorical columns
numeric_cols     = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Never impute the target
numeric_cols = [c for c in numeric_cols if c != 'TARGET']

# Find columns that actually have NaN
numeric_nan = [c for c in numeric_cols if df[c].isnull().sum() > 0]
cat_nan     = [c for c in categorical_cols if df[c].isnull().sum() > 0]

print(f"Numeric columns with NaN    : {len(numeric_nan)}")
print(f"Categorical columns with NaN: {len(cat_nan)}")

# Fill numeric with median
for col in numeric_nan:
    median_val = df[col].median()
    df[col]    = df[col].fillna(median_val)
    print(f"  {col:<30} filled with median = {median_val:.4f}")

# Fill categorical with UNKNOWN
for col in cat_nan:
    df[col] = df[col].fillna('UNKNOWN')
    print(f"  {col:<30} filled with UNKNOWN")

print(f"\nTotal NaN remaining : {df.isnull().sum().sum()}")
print(f"Shape               : {df.shape}")

Numeric columns with NaN    : 24
Categorical columns with NaN: 3
  AMT_ANNUITY                    filled with median = 24903.0000
  AMT_GOODS_PRICE                filled with median = 450000.0000
  CNT_FAM_MEMBERS                filled with median = 2.0000
  YEARS_BEGINEXPLUATATION_AVG    filled with median = 0.9816
  FLOORSMAX_AVG                  filled with median = 0.1667
  YEARS_BEGINEXPLUATATION_MODE   filled with median = 0.9816
  FLOORSMAX_MODE                 filled with median = 0.1667
  YEARS_BEGINEXPLUATATION_MEDI   filled with median = 0.9816
  FLOORSMAX_MEDI                 filled with median = 0.1667
  TOTALAREA_MODE                 filled with median = 0.0688
  OBS_30_CNT_SOCIAL_CIRCLE       filled with median = 0.0000
  DEF_30_CNT_SOCIAL_CIRCLE       filled with median = 0.0000
  OBS_60_CNT_SOCIAL_CIRCLE       filled with median = 0.0000
  DEF_60_CNT_SOCIAL_CIRCLE       filled with median = 0.0000
  DAYS_LAST_PHONE_CHANGE         filled with median = -757.0000
  AMT_RE

In [10]:
# Cell 7
cols_to_drop = [
    'YEARS_BEGINEXPLUATATION_AVG',
    'YEARS_BEGINEXPLUATATION_MODE',
    'YEARS_BEGINEXPLUATATION_MEDI',
    'FLOORSMAX_AVG',
    'FLOORSMAX_MODE',
    'FLOORSMAX_MEDI',
    'TOTALAREA_MODE',
]

df = df.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} low-signal columns")
print(f"Shape : {df.shape}")

Dropped 7 low-signal columns
Shape : (307511, 79)


In [11]:
# Cell 8 - Encode categorical columns

# Check what categorical columns we have
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns to encode: {len(categorical_cols)}")
for col in categorical_cols:
    print(f"  {col:<35} {df[col].nunique()} unique values")

Categorical columns to encode: 13
  NAME_CONTRACT_TYPE                  2 unique values
  CODE_GENDER                         3 unique values
  FLAG_OWN_CAR                        2 unique values
  FLAG_OWN_REALTY                     2 unique values
  NAME_TYPE_SUITE                     8 unique values
  NAME_INCOME_TYPE                    8 unique values
  NAME_EDUCATION_TYPE                 5 unique values
  NAME_FAMILY_STATUS                  6 unique values
  NAME_HOUSING_TYPE                   6 unique values
  OCCUPATION_TYPE                     19 unique values
  WEEKDAY_APPR_PROCESS_START          7 unique values
  ORGANIZATION_TYPE                   58 unique values
  EMERGENCYSTATE_MODE                 3 unique values


In [12]:
# Cell 9 - apply encoding

# ── Fix XNA values before encoding ─────────────────────────────────────────
df['CODE_GENDER']       = df['CODE_GENDER'].replace('XNA', 'UNKNOWN')
df['ORGANIZATION_TYPE'] = df['ORGANIZATION_TYPE'].replace('XNA', 'UNKNOWN')

# ── Binary columns — direct 0/1 map ────────────────────────────────────────
binary_mappings = {
    'NAME_CONTRACT_TYPE' : {'Cash loans': 0, 'Revolving loans': 1},
    'FLAG_OWN_CAR'       : {'N': 0, 'Y': 1},
    'FLAG_OWN_REALTY'    : {'N': 0, 'Y': 1},
}

for col, mapping in binary_mappings.items():
    df[col] = df[col].map(mapping)
    print(f"  {col:<25} → encoded as 0/1")

# ── One-Hot Encoding for remaining categoricals ─────────────────────────────
ohe_cols = [
    'CODE_GENDER', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START',
    'ORGANIZATION_TYPE', 'EMERGENCYSTATE_MODE'
]

df = pd.get_dummies(df, columns=ohe_cols, dtype=int)

# ── Drop SK_ID_CURR — just a row ID, no predictive value ───────────────────
df = df.drop(columns=['SK_ID_CURR'])

# ── Verify ──────────────────────────────────────────────────────────────────
print(f"\nShape after encoding  : {df.shape}")
print(f"Text columns remaining: {df.select_dtypes(include='object').shape[1]}")
print(f"Total NaN remaining   : {df.isnull().sum().sum()}")
print(f"\nTarget distribution:")
print(f"  0 (repaid)   : {(df['TARGET']==0).sum():,}")
print(f"  1 (defaulted): {(df['TARGET']==1).sum():,}")

  NAME_CONTRACT_TYPE        → encoded as 0/1
  FLAG_OWN_CAR              → encoded as 0/1
  FLAG_OWN_REALTY           → encoded as 0/1

Shape after encoding  : (307511, 191)
Text columns remaining: 0
Total NaN remaining   : 0

Target distribution:
  0 (repaid)   : 282,686
  1 (defaulted): 24,825


In [14]:
# Cell 10 - save processed data

output_path = '../data/application_train_processed.csv'

df.to_csv(output_path, index=False)

print(f"Dataset saved ✅")
print(f"  Path : {output_path}")
print(f"  Rows : {df.shape[0]:,}")
print(f"  Cols : {df.shape[1]}")

Dataset saved ✅
  Path : ../data/application_train_processed.csv
  Rows : 307,511
  Cols : 191


Check

In [15]:
df = pd.read_csv('/Users/geetharajamanickam/Desktop/creditguard/data/application_train_processed.csv')
df.head(5)

,TARGET,NAME_CONTRACT_TYPE,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_REGISTRATION,DAYS_ID_PUBLISH,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,...,ORGANIZATION_TYPE_Government,ORGANIZATION_TYPE_Hotel,ORGANIZATION_TYPE_Housing,ORGANIZATION_TYPE_Industry: type 1,ORGANIZATION_TYPE_Industry: type 10,ORGANIZATION_TYPE_Industry: type 11,ORGANIZATION_TYPE_Industry: type 12,ORGANIZATION_TYPE_Industry: type 13,ORGANIZATION_TYPE_Industry: type 2,ORGANIZATION_TYPE_Industry: type 3,ORGANIZATION_TYPE_Industry: type 4,ORGANIZATION_TYPE_Industry: type 5,ORGANIZATION_TYPE_Industry: type 6,ORGANIZATION_TYPE_Industry: type 7,ORGANIZATION_TYPE_Industry: type 8,ORGANIZATION_TYPE_Industry: type 9,ORGANIZATION_TYPE_Insurance,ORGANIZATION_TYPE_Kindergarten,ORGANIZATION_TYPE_Legal Services,ORGANIZATION_TYPE_Medicine,ORGANIZATION_TYPE_Military,ORGANIZATION_TYPE_Mobile,ORGANIZATION_TYPE_Other,ORGANIZATION_TYPE_Police,ORGANIZATION_TYPE_Postal,ORGANIZATION_TYPE_Realtor,ORGANIZATION_TYPE_Religion,ORGANIZATION_TYPE_Restaurant,ORGANIZATION_TYPE_School,ORGANIZATION_TYPE_Security,ORGANIZATION_TYPE_Security Ministries,ORGANIZATION_TYPE_Self-employed,ORGANIZATION_TYPE_Services,ORGANIZATION_TYPE_Telecom,ORGANIZATION_TYPE_Trade: type 1,ORGANIZATION_TYPE_Trade: type 2,ORGANIZATION_TYPE_Trade: type 3,ORGANIZATION_TYPE_Trade: type 4,ORGANIZATION_TYPE_Trade: type 5,ORGANIZATION_TYPE_Trade: type 6,ORGANIZATION_TYPE_Trade: type 7,ORGANIZATION_TYPE_Transport: type 1,ORGANIZATION_TYPE_Transport: type 2,ORGANIZATION_TYPE_Transport: type 3,ORGANIZATION_TYPE_Transport: type 4,ORGANIZATION_TYPE_UNKNOWN,ORGANIZATION_TYPE_University,EMERGENCYSTATE_MODE_No,EMERGENCYSTATE_MODE_UNKNOWN,EMERGENCYSTATE_MODE_Yes
0,1,0,0,1,0,202500.0,406597.5,24700.5,351000.0,0.018801,-3648.0,-2120,1,1,0,1,1,0,1.0,2,2,10,0,0,0,0,0,0,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
1,0,0,0,0,0,270000.0,1293502.5,35698.5,1129500.0,0.003541,-1186.0,-291,1,1,0,1,1,0,2.0,1,1,11,0,0,0,0,0,0,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
2,0,1,1,1,0,67500.0,135000.0,6750.0,135000.0,0.010032,-4260.0,-2531,1,1,1,1,1,0,1.0,2,2,9,0,0,0,0,0,0,0.0,0.0,0.0,0.0,-815.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
3,0,0,0,1,0,135000.0,312682.5,29686.5,297000.0,0.008019,-9833.0,-2437,1,1,0,1,0,0,2.0,2,2,17,0,0,0,0,0,0,2.0,0.0,2.0,0.0,-617.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,0,0,0,1,0,121500.0,513000.0,21865.5,513000.0,0.028663,-4311.0,-3458,1,1,0,1,0,0,1.0,2,2,11,0,0,0,0,1,1,0.0,0.0,0.0,0.0,-1106.0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0


Step 1 → Loaded raw data          : 122 columns

Step 2 → Dropped >50% missing     : 122 → 82 columns

Step 3 → EXT_SOURCE aggregation   : created mean_ext_score, is_missing_ext

Step 4 → Fixed DAYS_EMPLOYED      : 365243 → 0 for pensioners/unemployed

Step 5 → Created new features     : AGE_YEARS, EMPLOYMENT_YEARS, ratios etc.

Step 6 → Imputed remaining NaN    : median for numeric, UNKNOWN for categorical

Step 7 → Dropped low signal cols  : 86 → 79 columns

Step 8 → Encoded categoricals     : 79 → 191 columns (OHE expanded)

Step 9 → Saved processed dataset  : 191 columns, 307,511 rows